# 03 — Feature Engineering (2023–2025 data)
**Input:** `data/Cleaned/aqi_2023_2025_clean.csv`  **Output:** `data/Cleaned/aqi_2023_25_features.csv`

## The prediction task
`AQI_Bucket` is computed directly from `AQI`, so predicting *today's* bucket from *today's* AQI is trivial. We predict **tomorrow** instead:

| Task | Target | Metrics |
|---|---|---|
| A. Classification | tomorrow's `AQI_Bucket` | per-class precision, recall, F1, macro-F1, confusion matrix |
| B. Regression | tomorrow's `AQI` | MAE, RMSE, R² |

Your autocorrelation plot in the EDA justifies both: yesterday's AQI carries a lot of information about today's. This file has no pollutant concentrations, so we do **not** predict pollutants.

## Feature groups
| Group | Features |
|---|---|
| calendar | month, season, weekday, is_weekend, is_winter, is_festival (Diwali ±3 days) |
| lags | AQI_lag1, lag2, lag3, lag7 |
| rolling | 3, 7 and 14-day mean and 7-day std, **shifted by one day** so today is not included |
| context | state, number_of_monitoring_stations, today's and yesterday's dominant pollutant |
| city | city statistics computed on training data only (done in notebook 04) |

**Golden rule:** every feature must be known *today*. Nothing from tomorrow may leak in.

In [1]:
import pandas as pd
import numpy as np
import os, sys
sys.path.append("..")
from src.aqi_common import ALL_FEATURES, FEATURE_GROUPS

pd.set_option('display.max_columns', None)
IN_PATH = "../data/Cleaned/aqi_2023_2025_clean.csv"
OUT_PATH = "../data/Cleaned/aqi_2023_25_features.csv"


## Step 1 — Load and tidy the cleaned file
Works whether your cleaned file kept the raw Kaggle column names or already uses `City`, `AQI`, `AQI_Bucket`. It also tries both date formats.

In [2]:
raw = pd.read_csv(IN_PATH)
df = raw.rename(columns={'area': 'City', 'aqi_value': 'AQI', 'air_quality_status': 'AQI_Bucket', 'date': 'Date'})

d = pd.to_datetime(df['Date'], format='%Y-%m-%d', errors='coerce')
if d.isna().mean() > 0.5:
    d = pd.to_datetime(df['Date'], format='%d-%m-%Y', errors='coerce')
df['Date'] = d
df['AQI'] = pd.to_numeric(df['AQI'], errors='coerce')

for col, default in [('state', 'Unknown'), ('number_of_monitoring_stations', np.nan), ('prominent_pollutants', np.nan)]:
    if col not in df.columns:
        df[col] = default

df = df.dropna(subset=['City', 'Date', 'AQI', 'AQI_Bucket'])
df['City'] = df['City'].astype(str).str.strip().str.title()
df['AQI_Bucket'] = df['AQI_Bucket'].astype(str).str.strip().str.title()
df = df.drop_duplicates(subset=['City', 'Date'])

df = df[['City', 'state', 'Date', 'AQI', 'AQI_Bucket', 'number_of_monitoring_stations', 'prominent_pollutants']]
df = df.sort_values(['City', 'Date']).reset_index(drop=True)
print("Rows:", len(df), "| cities:", df['City'].nunique(), "| dates:", df['Date'].min().date(), "to", df['Date'].max().date())
df.head()

Rows: 235005 | cities: 291 | dates: 2022-04-01 to 2025-04-30


,City,state,Date,AQI,AQI_Bucket,number_of_monitoring_stations,prominent_pollutants
0,Agartala,Tripura,2022-04-01,78,Satisfactory,2,"PM10,SO2"
1,Agartala,Tripura,2022-04-02,77,Satisfactory,2,"PM10,SO2"
2,Agartala,Tripura,2022-04-03,76,Satisfactory,2,"PM2.5,PM10"
3,Agartala,Tripura,2022-04-04,57,Satisfactory,2,PM10
4,Agartala,Tripura,2022-04-05,64,Satisfactory,2,PM10


## Step 2 — Trap 1: date gaps
`shift(1)` moves by **row**, not by **day**. If a city is missing a day, "lag 1" would silently be 2 days old. Check how many gaps exist.

In [3]:
gap = df.groupby('City')['Date'].diff().dt.days.dropna()
print("Days between consecutive rows (should mostly be 1):")
print(gap.value_counts().sort_index().head(10))
print("\nShare of consecutive rows exactly 1 day apart:", round((gap == 1).mean() * 100, 2), "%")

Days between consecutive rows (should mostly be 1):
Date
1.0     219804
2.0       9241
3.0       2706
4.0       1143
5.0        582
6.0        336
7.0        210
8.0        138
9.0         97
10.0        75
Name: count, dtype: int64

Share of consecutive rows exactly 1 day apart: 93.65 %
